# 고객 세분화 및 타겟 마케팅
- 고객의 구매 빈도와 평균 주문 가치를 기준으로 네 가지 세그먼트로 분류한 경우, 각 세그먼트에 맞는 맞춤형 마케팅 전략을 적용할 수 있음
    - High Value - High Frequency
    - High Value - Low Frequency
    - Low Value - High Frequency
    - Low Value - Low Frequency

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# MySQL 데이터베이스 연결 설정
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='비밀번호',  # 실제 비밀번호로 변경하세요
    database='classicmodels'
)

In [ ]:
cursor = conn.cursor()

In [ ]:
# 각 고객(customerNumber)의 주문 횟수(orderCount)와 평균 주문 금액(avgOrderValue)를 계산하여, 가장 많은 주문을 한 고객부터 정렬하는 분석용 쿼리
# 즉, 어떤 고객이 가장 자주 주문했으며, 평균적으로 얼마나 많은 금액을 주문했는지 분석하는 것이 목적
query = """
SELECT 
    c.customerNumber, 
    COUNT(o.orderNumber) AS orderCount, 
    AVG(od.quantityOrdered * od.priceEach) AS avgOrderValue  -- 해당 고객의 평균 주문 금액
FROM 
    customers c  
JOIN 
    orders o ON c.customerNumber = o.customerNumber  
JOIN 
    orderdetails od ON o.orderNumber = od.orderNumber  
GROUP BY 
    c.customerNumber                                -- 고객별로 그룹화하여 총 주문 수 및 평균 주문 금액 계산
ORDER BY 
    orderCount DESC, avgOrderValue DESC;            -- 주문 개수가 많은 고객부터 정렬, 주문 개수가 같다면 평균 주문 금액이 높은 순서로 정렬

"""

In [ ]:
cursor.execute(query)

In [ ]:
# 결과를 DataFrame으로 변환
columns = [desc[0] for desc in cursor.description]
customer_df = pd.DataFrame(cursor.fetchall(), columns=columns)

In [ ]:
# 연결 종료
cursor.close()
conn.close()

In [ ]:
# 세분화 기준 정의 (임의로 설정)
orderCount_median = customer_df['orderCount'].median()
avgOrderValue_median = customer_df['avgOrderValue'].median()

In [ ]:
# 세분화 레이블링
customer_df['Segment'] = np.where((customer_df['orderCount'] >= orderCount_median) & (customer_df['avgOrderValue'] >= avgOrderValue_median), 'High Value - High Frequency',
                                  np.where((customer_df['orderCount'] < orderCount_median) & (customer_df['avgOrderValue'] >= avgOrderValue_median), 'High Value - Low Frequency',
                                           np.where((customer_df['orderCount'] >= orderCount_median) & (customer_df['avgOrderValue'] < avgOrderValue_median), 'Low Value - High Frequency', 'Low Value - Low Frequency')))

In [ ]:
# 세그먼트별 색상 매핑
color_map = {
    'High Value - High Frequency': 'green',
    'High Value - Low Frequency': 'blue',
    'Low Value - High Frequency': 'orange',
    'Low Value - Low Frequency': 'red'
}

In [ ]:
# 산점도 시각화
plt.figure(figsize=(10, 8))
sns.scatterplot(data=customer_df, x='orderCount', y='avgOrderValue', hue='Segment', palette=color_map, s=100)
plt.title('Customer Segmentation based on Order Count and Average Order Value')
plt.xlabel('Order Count')
plt.ylabel('Average Order Value')
plt.legend(title='Segment', title_fontsize='13', labelspacing=1.2)
plt.grid(True)
plt.show()

## 그룹 별 마케팅
- High Value - High Frequency (HVHF)
    - 특징: 이 그룹의 고객들은 높은 주문 가치와 빈번한 구매 행동을 보임.
    - 추천 마케팅 전략: 로열티 프로그램: 이들은 이미 브랜드에 충성도가 높은 고객군. 로열티 포인트, VIP 서비스 등을 제공하여 이들의 충성도를 더욱 강화.
    - 업셀링 및 크로스셀링: 관련 제품이나 더 고가의 제품을 제안하여 구매를 유도.
    - 개인화된 커뮤니케이션: 이메일 마케팅, SMS 등을 통해 개인화된 제안과 정보를 제공.

- High Value - Low Frequency (HVLF)
    - 특징: 높은 주문 가치를 보이지만 구매 빈도는 낮은 고객군.
    - 추천 마케팅 전략: 재구매 유도: 할인 쿠폰, 한정 판매 제품 정보 등을 제공하여 재구매를 유도.
    - 이벤트 초대: 특별 이벤트나 신제품 출시 행사에 이들을 초대하여 관심을 유도.
    - 피드백 요청: 제품이나 서비스에 대한 피드백을 요청하여 관계를 강화하고, 개선점을 찾음.

- Low Value - High Frequency (LVHF)
    - 특징: 자주 구매하지만 주문 가치는 낮은 고객군.
    - 추천 마케팅 전략: 가치 제안 강화: 더 높은 가치의 제품을 합리적인 가격에 제안하여 평균 주문 가치를 높임.
    - 번들 판매: 관련 제품을 함께 묶어서 판매하여 더 높은 가치의 구매를 유도.
    - 교육 및 정보 제공: 제품 사용법, 관련 정보를 제공하여 제품에 대한 인식을 높이고, 더 높은 가치 제품으로의 이동을 유도.
    
- Low Value - Low Frequency (LVLF)
    - 특징: 낮은 주문 가치와 드문 구매 행동을 보이는 고객군.
    - 추천 마케팅 전략: 인식 향상 캠페인: 브랜드 인식을 높이기 위한 마케팅 캠페인을 진행.
    - 입문 제품 제안: 저렴한 가격의 입문 제품을 제안하여 제품에 대한 관심을 유도.
    - 소셜 미디어 마케팅: 소셜 미디어를 통해 브랜드와 제품에 대한 인식을 증가.
    
